### ==============================================================================
### 🚀 THE ULTIMATE TRAFFIC DEMAND PREDICTION PIPELINE (100% EFFICIENCY BUILD)
### Models: CatBoost + LightGBM + XGBoost (Weighted Stacking)
### Features: Cyclical Time, Geo-Spatial Clustering, Target Encoding, Interactions
### ==============================================================================

In [3]:
# !pip install catboost lightgbm xgboost geohash2 shap scikit-learn pandas numpy

In [15]:
import pandas as pd
import numpy as np
import warnings
import geohash2
import os


import lightgbm as lgb 
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

## ==============================================================================
## 1. LOAD DATA & INITIAL SETUP
## ==============================================================================


## 1. Data Loading & Initial Setup
**Objective:** Load the raw data and prepare a unified pipeline for seamless processing.

In this section, we load the `train.csv` and `test.csv` datasets. A crucial step here is concatenating both datasets into a single dataframe (`df`) before feature engineering. 
* **Why do this?** Combining the data ensures that any transformations, missing value imputations, or categorical creations are applied identically to both the training and testing sets. This prevents discrepancies or "unseen category" errors later during model prediction. 
* We also safely decode the alphanumeric `geohash` strings into precise numerical `latitude` and `longitude` coordinates using the `geohash2` library.


In [16]:

print("⏳ [1/6] Loading Datasets...")

def decode_geohash(gh):
    """Safely decodes geohash to latitude and longitude."""
    try:
        lat, lon = geohash2.decode(gh)
        return float(lat), float(lon)
    except:
        return np.nan, np.nan

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

test_indices = test['Index'].copy()

train['is_train'] = 1
test['is_train'] = 0
test['demand'] = np.nan 
df = pd.concat([train, test], ignore_index=True)

⏳ [1/6] Loading Datasets...


## ==============================================================================
## 2. ADVANCED FEATURE ENGINEERING (SPATIO-TEMPORAL & INTERACTIONS)
## ==============================================================================

## 2. Advanced Feature Engineering (Spatio-Temporal & Interactions)
**Objective:** Extract hidden patterns from time, geography, and road conditions.

Machine learning models cannot automatically interpret standard timestamps or raw categorical strings. We engineer high-dimensional features to "teach" the model context:
* **Cyclical Time:** Standard time treats `23:59` and `00:01` as absolute opposites. By applying Sine and Cosine transformations (`sin_time`, `cos_time`), we map time onto a circle, allowing the model to understand continuous time flow.
* **Geospatial Clustering:** We extract the 4-character and 5-character prefixes of the `geohash` to group nearby locations into broader regional clusters, helping the model generalize for areas with sparse data.
* **Golden Interactions:** Traffic demand often depends on specific combinations (e.g., Heavy traffic on a *Residential Road* only at *8 AM*). By combining features like `RoadType + Hour` and `Geohash + LargeVehicles`, we create powerful interaction variables that drastically boost tree-based models.

In [17]:

print("⚙️ [2/6] Engineering High-Dimensional Features...")

# --- A. Time Features ---
df['hour'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else 0)
df['minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]) if pd.notnull(x) else 0)

# Flag high-traffic peak hours
df['is_peak_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df['day_of_week'] = df['day'] % 7
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Continuous & Cyclical Time (Crucial for ML to understand time flow)
df['continuous_time'] = 24 * 60 * (df['day'] - 1) + 60 * df['hour'] + df['minute']
df['time_in_mins'] = df['hour'] * 60 + df['minute']
df['sin_time'] = np.sin(2 * np.pi * df['time_in_mins'] / 1440)
df['cos_time'] = np.cos(2 * np.pi * df['time_in_mins'] / 1440)

# --- B. Geospatial Features ---
decoded = df['geohash'].apply(decode_geohash)
df['latitude'] = [x[0] for x in decoded]
df['longitude'] = [x[1] for x in decoded]
# Regional clustering via geohash prefixes
df['geohash_pref4'] = df['geohash'].str[:4]
df['geohash_pref5'] = df['geohash'].str[:5]

# --- C. Intelligent Imputation ---
# Temperature varies by region and time of day. Impute based on Grouped Median.
df['Temperature'] = df['Temperature'].fillna(
    df.groupby(['geohash_pref4', 'hour'])['Temperature'].transform('median')
)
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median()) # Global Fallback

# Fill Missing Categoricals
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in cat_cols:
    df[col] = df[col].fillna("Unknown").astype(str)

# --- D. Golden Interaction Features ---
df['geo_road'] = df['geohash'].astype(str) + "_" + df['RoadType'].astype(str)
df['road_hour'] = df['RoadType'].astype(str) + "_" + df['hour'].astype(str)
df['geo_vehicle'] = df['geohash'].astype(str) + "_" + df['LargeVehicles'].astype(str)
df['weather_road'] = df['Weather'].astype(str) + "_" + df['RoadType'].astype(str)

⚙️ [2/6] Engineering High-Dimensional Features...


## ==============================================================================
## 3. OUT-OF-FOLD (OOF) TARGET ENCODING (PREVENTS DATA LEAKAGE)
## ==============================================================================

## 3. Leak-Free Out-Of-Fold (OOF) Target Encoding
**Objective:** Replace complex categorical variables with historical demand averages without causing Data Leakage.

Target Encoding is one of the most powerful techniques for tabular data. It replaces a category (like a specific `geohash`) with the average target (`demand`) of that category. 
* **The Problem:** Doing this directly on the whole dataset causes severe **Data Leakage** (the model memorizes the answer and overfits).
* **The Solution (OOF):** We use a K-Fold cross-validation strategy strictly on the training data. The model calculates the historical average for a category using 4 folds, and maps that average onto the 1 unseen fold. This ensures the model learns the *general trend* rather than memorizing the exact answers, maintaining perfect integrity for the final test predictions.

In [7]:

print("🛡️ [3/6] Applying Leak-Free OOF Target Encoding...")

train_df = df[df['is_train'] == 1].copy()
test_df = df[df['is_train'] == 0].copy()

def oof_target_encoding(train_df, test_df, columns, target_col='demand'):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for col in columns:
        new_col_name = f"{col}_target_enc"
        train_df[new_col_name] = np.nan
        test_df[new_col_name] = np.nan
        
        for train_idx, val_idx in kf.split(train_df):
            X_tr, X_val = train_df.iloc[train_idx], train_df.iloc[val_idx]
            mean_enc = X_tr.groupby(col)[target_col].mean()
            train_df.loc[val_idx, new_col_name] = train_df.loc[val_idx, col].map(mean_enc)
            
        global_mean_enc = train_df.groupby(col)[target_col].mean()
        test_df[new_col_name] = test_df[col].map(global_mean_enc)
        
        global_mean = train_df[target_col].mean()
        train_df[new_col_name].fillna(global_mean, inplace=True)
        test_df[new_col_name].fillna(global_mean, inplace=True)
        
    return train_df, test_df

target_encode_cols = ['geohash', 'road_hour', 'geo_road', 'geohash_pref5']
train_df, test_df = oof_target_encoding(train_df, test_df, target_encode_cols, 'demand')

🛡️ [3/6] Applying Leak-Free OOF Target Encoding...


## ==============================================================================
## 4. PREPARING MODELS FOR THE TRIPLE ENSEMBLE
## ==============================================================================


## 4. Initializing the Triple Ensemble
**Objective:** Prepare the data types and hyperparameters for three state-of-the-art Gradient Boosting algorithms.

No single model is perfect. To achieve maximum accuracy, we rely on a trio of powerful algorithms:
1. **CatBoost:** Exceptionally strong at handling high-cardinality categorical features (like our interaction variables).
2. **LightGBM:** Fast, memory-efficient, and highly sensitive to numerical depth and spatial patterns.
3. **XGBoost:** The industry standard for tabular data, excellent at strictly minimizing prediction errors (RMSE).

In this step, we explicitly cast our final feature columns into the `string` and `category` data types natively required by these distinct algorithms. Hyperparameters (like `learning_rate`, `depth`, and `n_estimators`) have been carefully tuned for CPU efficiency and peak validation accuracy.

In [18]:

print("🧠 [4/6] Initializing Models...")

drop_cols = ['is_train', 'Index', 'timestamp', 'demand']
features = [c for c in train_df.columns if c not in drop_cols]

X = train_df[features].copy()
y = train_df['demand']
X_test = test_df[features].copy()

final_cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 
                  'geohash_pref4', 'geohash_pref5', 'geo_road', 'road_hour', 
                  'geo_vehicle', 'weather_road']

for col in final_cat_cols:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

X_tree = X.copy()
X_test_tree = X_test.copy()
for col in final_cat_cols:
    X_tree[col] = X_tree[col].astype('category')
    X_test_tree[col] = X_test_tree[col].astype('category')

cb_params = {'iterations': 2000, 'learning_rate': 0.04, 'depth': 8, 'l2_leaf_reg': 3, 
             'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'random_seed': 42, 'verbose': False}

lgb_params = {'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 8, 
              'objective': 'regression', 'metric': 'rmse', 'random_state': 42, 
              'verbose': -1, 'n_jobs': -1}

xgb_params = {'n_estimators': 1500, 'learning_rate': 0.03, 'max_depth': 7, 
              'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'random_state': 42, 
              'enable_categorical': True, 'tree_method': 'hist', 'n_jobs': -1,
              'early_stopping_rounds': 100}

🧠 [4/6] Initializing Models...


# ==============================================================================
# 5. K-FOLD CROSS VALIDATION & STACKING
# ==============================================================================

## 5. K-Fold Cross Validation & Model Stacking
**Objective:** Train robust models on diverse data splits and blend their predictions to eliminate variance.

Instead of training a model once, we use **5-Fold Cross Validation**. The data is split into 5 parts; the models train on 4 parts and are evaluated on the 5th, rotating until all data has been evaluated.
* **Why K-Fold?** It ensures our model performs consistently across different geographic and temporal distributions, guaranteeing that our local RMSE score will closely match the final public Leaderboard score.
* **Weighted Ensembling (Stacking):** For every fold, we generate predictions from all three models. We then blend them using a mathematically stable weighted average: **(40% CatBoost + 40% LightGBM + 20% XGBoost)**. This balances the unique strengths of each algorithm and creates a remarkably stable, highly accurate final prediction.

In [19]:

print("🔥 [5/6] Training Triple Ensemble (5-Fold CV)... This will take a few minutes.\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cat_test_preds = np.zeros(len(test_df))
lgb_test_preds = np.zeros(len(test_df))
xgb_test_preds = np.zeros(len(test_df))
oof_blend = np.zeros(len(train_df))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Training Fold {fold + 1}/5 ---")
    y_tr, y_va = y.iloc[tr_idx], y.iloc[val_idx]
    
    # 1. CatBoost
    cat_model = CatBoostRegressor(**cb_params)
    cat_model.fit(X.iloc[tr_idx], y_tr, eval_set=(X.iloc[val_idx], y_va), 
                  cat_features=final_cat_cols, early_stopping_rounds=100)
    cat_va_pred = cat_model.predict(X.iloc[val_idx])
    cat_test_preds += cat_model.predict(X_test) / 5
    
    # 2. LightGBM
    lgb_model = LGBMRegressor(**lgb_params)
    lgb_model.fit(X_tree.iloc[tr_idx], y_tr, eval_set=[(X_tree.iloc[val_idx], y_va)], 
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_va_pred = lgb_model.predict(X_tree.iloc[val_idx])
    lgb_test_preds += lgb_model.predict(X_test_tree) / 5
    
    # 3. XGBoost
    xgb_model = XGBRegressor(**xgb_params)
    xgb_model.fit(X_tree.iloc[tr_idx], y_tr, eval_set=[(X_tree.iloc[val_idx], y_va)], 
                  verbose=False)
    xgb_va_pred = xgb_model.predict(X_tree.iloc[val_idx])
    xgb_test_preds += xgb_model.predict(X_test_tree) / 5
    
    fold_pred = (0.4 * cat_va_pred) + (0.4 * lgb_va_pred) + (0.2 * xgb_va_pred)
    oof_blend[val_idx] = fold_pred
    fold_rmse = np.sqrt(mean_squared_error(y_va, fold_pred))
    print(f"✅ Fold {fold + 1} Blended RMSE: {fold_rmse:.6f}\n")

final_rmse = np.sqrt(mean_squared_error(y, oof_blend))
print(f"🏆 FINAL OVERALL ENSEMBLE RMSE: {final_rmse:.6f}")

🔥 [5/6] Training Triple Ensemble (5-Fold CV)... This will take a few minutes.

--- Training Fold 1/5 ---
✅ Fold 1 Blended RMSE: 0.028002

--- Training Fold 2/5 ---
✅ Fold 2 Blended RMSE: 0.028577

--- Training Fold 3/5 ---
✅ Fold 3 Blended RMSE: 0.027860

--- Training Fold 4/5 ---
✅ Fold 4 Blended RMSE: 0.028566

--- Training Fold 5/5 ---
✅ Fold 5 Blended RMSE: 0.027760

🏆 FINAL OVERALL ENSEMBLE RMSE: 0.028155


# ==============================================================================
# 6. GENERATE FINAL SUBMISSION
# ==============================================================================

## 6. Post-Processing & Final Submission
**Objective:** Finalize predictions and format the output for competition evaluation.

After aggregating the blended predictions across all 5 folds for the test set, we perform one final mathematical post-processing step: **Clipping**.
* Since "Traffic Demand" represents physical volume, it is logically impossible for demand to be a negative number. However, regression models can occasionally predict slightly negative values for extremely sparse areas. We use `.clip(lower=0)` to convert any negative anomalies to zero.
* Finally, we map the predicted demand to the original test `Index` and export our highly optimized `ultimate_ensemble_submission.csv` file.

In [20]:


print("📄 [6/6] Generating Final Output...")

final_test_preds = (0.4 * cat_test_preds) + (0.4 * lgb_test_preds) + (0.2 * xgb_test_preds)

submission = pd.DataFrame({
    "Index": test_indices,
    "demand": final_test_preds
})

submission['demand'] = submission['demand'].clip(lower=0)
submission.to_csv("ultimate_ensemble_submission.csv", index=False)
print("🎉 'ultimate_ensemble_submission.csv' generated successfully! You are ready to submit.")

📄 [6/6] Generating Final Output...
🎉 'ultimate_ensemble_submission.csv' generated successfully! You are ready to submit.
